# AlphaZero Chess Training on Google Colab

This notebook trains a neural network to play chess using the AlphaZero algorithm.

## What This Does:
- Trains a ResNet-based neural network using self-play
- Uses Monte Carlo Tree Search (MCTS) for move selection
- Learns both move policy and position evaluation
- Exports trained model to ONNX format for browser deployment

## Training Time:
- **Quick Mode** (recommended): ~30-60 minutes on GPU
- **Full Mode**: ~5-7 hours on GPU

## Instructions:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run all cells in order
3. Download the trained model files at the end
4. Upload to your project

## 1. Setup and Dependencies

## 0. Environment Detection & Setup

Works with both:
- **Web Colab**: Traditional browser interface
- **VS Code Colab**: New VS Code extension (Nov 2025)

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running on Google Colab")
    print("  Environment: Web Colab or VS Code Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")

# Optional: Mount Google Drive for persistent storage
if IN_COLAB:
    mount_drive = input("Mount Google Drive for model persistence? (y/n): ").lower() == 'y'
    
    if mount_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        
        # Set save directory to Drive
        SAVE_DIR = '/content/drive/MyDrive/chess_models'
        import os
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"✓ Models will be saved to: {SAVE_DIR}")
    else:
        SAVE_DIR = '.'
        print("⚠️  Models will be saved to Colab runtime (temporary)")
        print("   Remember to download them before session ends!")
else:
    SAVE_DIR = '.'
    print("Models will be saved to current directory")

In [ ]:
# Install required packages
!pip install torch python-chess numpy onnx onnxruntime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess
import numpy as np
import os
import time
import math
from collections import deque
from typing import List, Tuple

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be VERY slow.")
    print("   Go to Runtime → Change runtime type → Select GPU")

## 2. Neural Network Architecture

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with batch normalization"""
    def __init__(self, channels=256):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = F.relu(out)
        return out


class AlphaZeroNet(nn.Module):
    """
    AlphaZero Neural Network
    
    Architecture:
    - Initial convolution block
    - Residual tower (6-19 blocks)
    - Policy head (move probabilities)
    - Value head (position evaluation)
    """
    def __init__(self, num_res_blocks=10, num_channels=128):
        super(AlphaZeroNet, self).__init__()
        
        # Input: 8x8x18 (board representation)
        # 12 piece types (6 per color) + 6 auxiliary planes
        self.input_channels = 18
        
        # Initial convolution block
        self.conv_block = nn.Sequential(
            nn.Conv2d(self.input_channels, num_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_channels),
            nn.ReLU()
        )
        
        # Residual tower
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(num_channels) for _ in range(num_res_blocks)]
        )
        
        # Policy head
        self.policy_conv = nn.Conv2d(num_channels, 32, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32 * 8 * 8, 4096)  # All possible moves
        
        # Value head
        self.value_conv = nn.Conv2d(num_channels, 32, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(32)
        self.value_fc1 = nn.Linear(32 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)
    
    def forward(self, x):
        # Initial conv
        x = self.conv_block(x)
        
        # Residual tower
        for block in self.res_blocks:
            x = block(x)
        
        # Policy head
        policy = F.relu(self.policy_bn(self.policy_conv(x)))
        policy = policy.view(-1, 32 * 8 * 8)
        policy = self.policy_fc(policy)
        policy = F.log_softmax(policy, dim=1)
        
        # Value head
        value = F.relu(self.value_bn(self.value_conv(x)))
        value = value.view(-1, 32 * 8 * 8)
        value = F.relu(self.value_fc1(value))
        value = torch.tanh(self.value_fc2(value))
        
        return policy, value

print("✓ Neural network architecture defined")

## 3. Board Encoding Functions

In [ ]:
def board_to_tensor(board: chess.Board) -> np.ndarray:
    """
    Convert chess board to neural network input tensor
    
    Returns: 8x8x18 numpy array
    - 12 channels for piece positions (6 per color)
    - 6 auxiliary channels (castling rights, en passant, turn, etc.)
    """
    tensor = np.zeros((18, 8, 8), dtype=np.float32)
    
    # Piece planes (12 channels)
    piece_idx = {'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
                 'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11}
    
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            rank, file = divmod(square, 8)
            channel = piece_idx[piece.symbol()]
            tensor[channel, rank, file] = 1.0
    
    # Auxiliary planes (6 channels)
    # Channel 12: White's turn
    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0
    
    # Channel 13-16: Castling rights
    if board.has_kingside_castling_rights(chess.WHITE):
        tensor[13, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE):
        tensor[14, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):
        tensor[15, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK):
        tensor[16, :, :] = 1.0
    
    # Channel 17: En passant
    if board.ep_square is not None:
        rank, file = divmod(board.ep_square, 8)
        tensor[17, rank, file] = 1.0
    
    return tensor


def move_to_index(move: chess.Move) -> int:
    """Convert chess move to policy index (0-4095)"""
    from_square = move.from_square
    to_square = move.to_square
    index = from_square * 64 + to_square
    return index


def index_to_move(index: int) -> Tuple[int, int]:
    """Convert policy index back to move coordinates"""
    from_square = index // 64
    to_square = index % 64
    return from_square, to_square

print("✓ Board encoding functions defined")

## 4. Monte Carlo Tree Search (MCTS)

In [ ]:
class MCTSNode:
    """MCTS node for AlphaZero self-play"""
    def __init__(self, board: chess.Board, parent=None, move=None, prior=0.0):
        self.board = board.copy()
        self.parent = parent
        self.move = move
        self.prior = prior
        
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0.0
        self.is_expanded = False
    
    def value(self):
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count
    
    def ucb_score(self, c_puct=1.0):
        """Upper Confidence Bound score"""
        if self.parent is None:
            return float('inf')
        
        u = c_puct * self.prior * math.sqrt(self.parent.visit_count) / (1 + self.visit_count)
        return self.value() + u
    
    def select_child(self, c_puct=1.0):
        """Select child with highest UCB score"""
        return max(self.children.values(), key=lambda child: child.ucb_score(c_puct))
    
    def expand(self, policy_probs):
        """Expand node with legal moves"""
        self.is_expanded = True
        
        for move in self.board.legal_moves:
            move_idx = move_to_index(move)
            prior = policy_probs[move_idx]
            
            child_board = self.board.copy()
            child_board.push(move)
            
            self.children[move] = MCTSNode(child_board, parent=self, move=move, prior=prior)
    
    def backup(self, value):
        """Backpropagate value up the tree"""
        self.visit_count += 1
        self.value_sum += value
        
        if self.parent:
            self.parent.backup(-value)  # Negate for opponent


def mcts_search(board: chess.Board, network: AlphaZeroNet, num_simulations=400, c_puct=1.0):
    """
    Perform MCTS search using neural network guidance
    
    Returns: policy vector (probability distribution over moves)
    """
    root = MCTSNode(board)
    
    device = next(network.parameters()).device
    
    # Get initial policy and value from network
    state_tensor = torch.FloatTensor(board_to_tensor(board)).unsqueeze(0).to(device)
    with torch.no_grad():
        policy_logits, _ = network(state_tensor)
        policy_probs = torch.exp(policy_logits).cpu().numpy()[0]
    
    root.expand(policy_probs)
    
    # Run simulations
    for _ in range(num_simulations):
        node = root
        
        # Selection
        while node.is_expanded and not node.board.is_game_over():
            node = node.select_child(c_puct)
        
        # Expansion and evaluation
        if not node.board.is_game_over():
            state_tensor = torch.FloatTensor(board_to_tensor(node.board)).unsqueeze(0).to(device)
            with torch.no_grad():
                policy_logits, value = network(state_tensor)
                policy_probs = torch.exp(policy_logits).cpu().numpy()[0]
                value = value.item()
            
            node.expand(policy_probs)
            node.backup(value)
        else:
            # Terminal node
            result = node.board.result()
            value = 1.0 if result == "1-0" else (-1.0 if result == "0-1" else 0.0)
            node.backup(value)
    
    # Return visit count distribution as policy
    policy = np.zeros(4096, dtype=np.float32)
    total_visits = sum(child.visit_count for child in root.children.values())
    
    for move, child in root.children.items():
        move_idx = move_to_index(move)
        policy[move_idx] = child.visit_count / total_visits if total_visits > 0 else 0.0
    
    return policy

print("✓ MCTS implementation defined")

## 5. Training Functions

In [ ]:
class ChessDataset(Dataset):
    """Dataset for chess positions"""
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        state, policy, value = self.data[idx]
        return torch.FloatTensor(state), torch.FloatTensor(policy), torch.FloatTensor([value])


def generate_selfplay_data(network: AlphaZeroNet, num_games=50, num_simulations=400):
    """Generate training data through self-play"""
    print(f"Generating {num_games} self-play games...")
    
    training_data = []
    device = next(network.parameters()).device
    
    start_time = time.time()
    
    for game_num in range(num_games):
        board = chess.Board()
        game_data = []
        
        while not board.is_game_over():
            # Get MCTS policy
            policy = mcts_search(board, network, num_simulations=num_simulations)
            
            # Store (state, policy, placeholder_value)
            state = board_to_tensor(board)
            game_data.append((state, policy))
            
            # Sample move from policy
            legal_moves = list(board.legal_moves)
            legal_indices = [move_to_index(move) for move in legal_moves]
            legal_probs = policy[legal_indices]
            legal_probs = legal_probs / legal_probs.sum()  # Normalize
            
            chosen_idx = np.random.choice(len(legal_moves), p=legal_probs)
            board.push(legal_moves[chosen_idx])
        
        # Get game result
        result = board.result()
        if result == "1-0":
            value = 1.0
        elif result == "0-1":
            value = -1.0
        else:
            value = 0.0
        
        # Assign values to all positions (alternate for each move)
        for i, (state, policy) in enumerate(game_data):
            position_value = value if i % 2 == 0 else -value
            training_data.append((state, policy, position_value))
        
        if (game_num + 1) % 5 == 0:
            elapsed = time.time() - start_time
            games_per_min = (game_num + 1) / (elapsed / 60)
            print(f"  Game {game_num + 1}/{num_games} | Result: {result} | {games_per_min:.1f} games/min")
    
    elapsed = time.time() - start_time
    print(f"✓ Generated {len(training_data)} training positions in {elapsed/60:.1f} minutes")
    return training_data


def train_network(network: AlphaZeroNet, training_data, epochs=10, batch_size=64, learning_rate=0.001):
    """Train the neural network"""
    print(f"Training network for {epochs} epochs...")
    
    device = next(network.parameters()).device
    optimizer = optim.Adam(network.parameters(), lr=learning_rate)
    
    dataset = ChessDataset(training_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    network.train()
    
    for epoch in range(epochs):
        total_loss = 0.0
        total_policy_loss = 0.0
        total_value_loss = 0.0
        
        for states, target_policies, target_values in dataloader:
            states = states.to(device)
            target_policies = target_policies.to(device)
            target_values = target_values.to(device)
            
            # Forward pass
            policy_logits, values = network(states)
            
            # Loss calculation
            policy_loss = -torch.mean(torch.sum(target_policies * policy_logits, dim=1))
            value_loss = F.mse_loss(values, target_values)
            loss = policy_loss + value_loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
        
        avg_loss = total_loss / len(dataloader)
        avg_policy_loss = total_policy_loss / len(dataloader)
        avg_value_loss = total_value_loss / len(dataloader)
        
        print(f"  Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} (Policy: {avg_policy_loss:.4f}, Value: {avg_value_loss:.4f})")

print("✓ Training functions defined")

## 6. Training Configuration

Choose your training mode:

In [ ]:
# ========================================
# CONFIGURATION - CHOOSE YOUR TRAINING MODE
# ========================================

# Set to True for quick training, False for full training
QUICK_MODE = True  # Change to False for full training

if QUICK_MODE:
    print("="*60)
    print("QUICK TRAINING MODE")
    print("="*60)
    config = {
        'num_iterations': 3,
        'num_selfplay_games': 10,
        'num_mcts_simulations': 100,
        'training_epochs': 5,
        'num_res_blocks': 6,
        'num_channels': 64,
        'batch_size': 64,
        'learning_rate': 0.001
    }
    print("Iterations: 3")
    print("Games per iteration: 10")
    print("MCTS simulations: 100")
    print("Network: 6 residual blocks, 64 channels")
    print("Estimated time: 30-60 minutes on GPU")
else:
    print("="*60)
    print("FULL TRAINING MODE")
    print("="*60)
    config = {
        'num_iterations': 10,
        'num_selfplay_games': 50,
        'num_mcts_simulations': 400,
        'training_epochs': 10,
        'num_res_blocks': 10,
        'num_channels': 128,
        'batch_size': 64,
        'learning_rate': 0.001
    }
    print("Iterations: 10")
    print("Games per iteration: 50")
    print("MCTS simulations: 400")
    print("Network: 10 residual blocks, 128 channels")
    print("Estimated time: 5-7 hours on GPU")

print("="*60)

## 7. Initialize Network

In [ ]:
# Create network
network = AlphaZeroNet(
    num_res_blocks=config['num_res_blocks'],
    num_channels=config['num_channels']
).to(device)

# Count parameters
num_params = sum(p.numel() for p in network.parameters())
print(f"Network initialized: {num_params:,} parameters")
print(f"Model size: ~{num_params * 4 / 1e6:.1f} MB")

## 8. Run Training

This will take some time. Go get a coffee! ☕

In [ ]:
# Create checkpoint directory in save location
checkpoint_dir = os.path.join(SAVE_DIR, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

# Training loop
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

training_start = time.time()

for iteration in range(config['num_iterations']):
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration + 1}/{config['num_iterations']}")
    print(f"{'='*60}")
    
    iter_start = time.time()
    
    # Generate self-play data
    training_data = generate_selfplay_data(
        network,
        num_games=config['num_selfplay_games'],
        num_simulations=config['num_mcts_simulations']
    )
    
    # Train network
    train_network(
        network,
        training_data,
        epochs=config['training_epochs'],
        batch_size=config['batch_size'],
        learning_rate=config['learning_rate']
    )
    
    # Save checkpoint to SAVE_DIR
    checkpoint_path = os.path.join(checkpoint_dir, f"iter_{iteration+1}.pt")
    torch.save({
        'iteration': iteration + 1,
        'model_state_dict': network.state_dict(),
        'config': config
    }, checkpoint_path)
    
    iter_elapsed = time.time() - iter_start
    print(f"✓ Iteration {iteration+1} completed in {iter_elapsed/60:.1f} minutes")
    print(f"✓ Checkpoint saved: {checkpoint_path}")

training_elapsed = time.time() - training_start

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)
print(f"Total training time: {training_elapsed/60:.1f} minutes ({training_elapsed/3600:.2f} hours)")

# Save final model to SAVE_DIR
final_path = os.path.join(SAVE_DIR, "alphazero_chess_final.pt")
torch.save(network.state_dict(), final_path)
print(f"✓ Final model saved: {final_path}")

## 9. Convert Model to ONNX Format

This converts the PyTorch model to ONNX for browser deployment.

In [ ]:
import torch.onnx

print("="*60)
print("Converting to ONNX Format")
print("="*60)

# Set model to evaluation mode
network.eval()

# Create dummy input (8x8x18 board tensor)
dummy_input = torch.randn(1, 18, 8, 8).to(device)

# Export to ONNX (save to SAVE_DIR)
onnx_path = os.path.join(SAVE_DIR, "alphazero_chess.onnx")
torch.onnx.export(
    network,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['policy', 'value'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'policy': {0: 'batch_size'},
        'value': {0: 'batch_size'}
    }
)

print(f"✓ ONNX model saved: {onnx_path}")

# Verify ONNX model
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model verification passed")
except Exception as e:
    print(f"⚠ ONNX verification warning: {e}")

# Check file sizes
pt_size = os.path.getsize(final_path) / 1e6
onnx_size = os.path.getsize(onnx_path) / 1e6

print(f"\nModel Sizes:")
print(f"  PyTorch (.pt): {pt_size:.1f} MB")
print(f"  ONNX (.onnx): {onnx_size:.1f} MB")

## 10. Download Trained Models

Download these files to your local machine and upload them to your project.

In [ ]:
import json

print("="*60)
print("Preparing files for download")
print("="*60)

# Save config file to SAVE_DIR
config_path = os.path.join(SAVE_DIR, "training_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\nTrained files location:")
print(f"  1. PyTorch model: {final_path}")
print(f"  2. ONNX model: {onnx_path}")
print(f"  3. Training config: {config_path}")
print(f"  4. Checkpoints: {checkpoint_dir}/")

# Download files if using web Colab
# VS Code Colab users can access files directly through file explorer
if IN_COLAB:
    try:
        from google.colab import files
        
        user_wants_download = input("\nDownload files now? (y/n - VS Code users can skip): ").lower() == 'y'
        
        if user_wants_download:
            print("\nDownloading files...")
            print("1. PyTorch model (for future training)")
            files.download(final_path)
            
            print("2. ONNX model (for browser deployment)")
            files.download(onnx_path)
            
            print("3. Training configuration")
            files.download(config_path)
        else:
            print("\n✓ Skipped download")
            print("  Files are saved and accessible via file explorer")
            if SAVE_DIR.startswith('/content/drive'):
                print(f"  Location: Google Drive → chess_models/")
    except ImportError:
        print("\n⚠️  Running locally - files already saved to disk")

print("\n" + "="*60)
print("ALL DONE!")
print("="*60)
print("\nNext steps:")
print("1. Upload the ONNX model to your project")
print("2. Convert ONNX → TensorFlow → TensorFlow.js (see instructions below)")
print("3. Integrate with your chess application")
print("\nConversion commands (run locally):")
print("  # Install tools")
print("  pip install onnx-tf tensorflowjs")
print("")
print("  # Convert ONNX to TensorFlow")
print("  onnx-tf convert -i alphazero_chess.onnx -o tensorflow_model")
print("")
print("  # Convert TensorFlow to TensorFlow.js")
print("  tensorflowjs_converter \\")
print("      --input_format=tf_saved_model \\")
print("      --output_format=tfjs_graph_model \\")
print("      tensorflow_model \\")
print("      public/tfjs_model")

## 11. Test the Model (Optional)

Quick test to make sure the model works.

In [ ]:
# Test inference
print("Testing model inference...")

network.eval()
test_board = chess.Board()

# Convert board to tensor
state_tensor = torch.FloatTensor(board_to_tensor(test_board)).unsqueeze(0).to(device)

# Get predictions
with torch.no_grad():
    policy_logits, value = network(state_tensor)
    policy_probs = torch.exp(policy_logits).cpu().numpy()[0]
    value = value.item()

print(f"✓ Model inference successful")
print(f"  Position value: {value:.3f}")
print(f"  Policy entropy: {-(policy_probs * np.log(policy_probs + 1e-10)).sum():.3f}")

# Find top moves
legal_moves = list(test_board.legal_moves)
move_probs = [(move, policy_probs[move_to_index(move)]) for move in legal_moves]
move_probs.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 5 recommended moves:")
for i, (move, prob) in enumerate(move_probs[:5]):
    print(f"  {i+1}. {move.uci()} (probability: {prob:.4f})")